# 第2段階　打牌前に選べた行動の測定
Q1→Q2→Q3→Q4→Q5 の順。Q2で対象件数とZIPを確認し、Q3で実牌譜の計数・打牌前手牌・処理秒数を確認します。中断後は同じ入力でQ1から実行すると検証済みバッチを再利用します。成果物は毎回新しい実行フォルダに保存します。

初回混合手牌、非現物ツモ時、形状テンパイを選べた時の3種類を数えます。単独リーチ者自身の河と、切る直前の自分の手牌を使用します。自分の成立前のリーチ宣言打牌は意思決定に含めます。カンの除外範囲は直前の自己打牌（局頭なら局開始）から今回の自己打牌まで。初回に機会がなくても、後の打牌への繰り越しはしません。

In [ ]:
# Q1　Drive接続、固定コードの取得、単体テスト
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, urllib.request, tempfile
drive.mount('/content/drive')
PIN = '90595d0c8b19bc77bdf7968d1e76d4c4a6117b5f'
src = Path(tempfile.mkdtemp(prefix='phase2_'))
paths = ['mahjong_research/'+f for f in ['__init__.py','inputs.py','tenhou.py','analysis.py','legacy.py']]
paths += ['smoothing/study.py','stability/stability.py'] + ['choices/'+f for f in
          ['choices.py','pipeline.py','test_choices.py']]
for rel in paths:
    dest = src / rel
    dest.parent.mkdir(exist_ok=True)
    url = f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{PIN}/research_v2/{rel}'
    with urllib.request.urlopen(url, timeout=60) as response:
        dest.write_bytes(response.read())
paths = [str(src/p) for p in ('choices','stability','smoothing','.')]
env = dict(os.environ, PYTHONPATH=os.pathsep.join(paths), MPLBACKEND='Agg')
subprocess.run([sys.executable,'-m','unittest','-q','test_choices'],cwd=src,env=env,check=True)
sys.path[:0] = paths
from pipeline import prepare, extract_batches, analyze
import pandas as pd
from IPython.display import display, Markdown

In [ ]:
# Q2　C5入力と元ZIPを照合し、今回の対象対局件数を表示
ROOT = Path('/content/drive/MyDrive/卒論')
RUN = ROOT / 'research_v2_runs/20261105T071722Z_5da6c9ad'
ZIP_OVERRIDE = None  # ZIPを移動した場合だけ、しょぼんｎのZIPのフルパスを指定
ctx = prepare(RUN, ROOT/'research_v2_choices', ZIP_OVERRIDE)
print('入力ZIP',ctx['archive'], '再開用キャッシュ',ctx['cache'])

In [ ]:
# Q3　低・高段位期から各2対局を先行照合。旧20項目の分子・分母もC5と比較
pilot = extract_batches(ctx,pilot_only=True)
display(pilot)
import json
events = json.loads((ctx['out']/'pilot_events.json').read_text())
display(pd.DataFrame(events).head(12))
print('機会は候補1・2・3の順。牌IDは136枚識別、riverとtenpai_discardsは34牌種コード。')

In [ ]:
# Q4　対象対局の追加カウンタを抽出。完了済みバッチは読み込み
records = extract_batches(ctx)
print(len(records),'対局を照合。保存先',ctx['out'])
display(records.filter(regex='__denominator$').sum())

In [ ]:
# Q5　機会数方式の10・20対局窓、段位群別の安定性、既存20項目との順位相関
decision = analyze(ctx,records)
display(decision)
display(Markdown((ctx['out']/'report_ja.md').read_text(encoding='utf-8')))
print('出力',ctx['out'])

**実データの確認箇所**　試験対局のイベントは `pilot_events.json`、対象対局全体の分子・分母は `new_game_features.csv`、元XML照合は `extraction_audit.csv`。失敗時は `extraction_failure.json` を確認します。

`group_statistics.csv` と `period_statistics.csv` に実測値、`stability_decisions.csv` に採否理由、`correlations.csv` に既存指標とのペアごとの有効窓数とSpearman相関、`window_values.csv` に元対局IDを保存します。安定性の比較基準は同じC5入力から再現した第1段階の既存率の基準値です。観測開始前の履歴と同時間帯内順序は未確認です。